# Radar FFT Cube Progress

这个 notebook 记录从 TI mmWave Studio/DCA1000 采集的 IWR6843 原始 ADC `.bin` 文件，到三次 FFT 生成 radar cube 并提取点云的完整处理流程。

核心链路：

1. 读取 mmWave Studio 导出的 raw ADC bin。
2. 按 DCA1000 LVDS 数据格式恢复 complex ADC samples。
3. 重排为 frame/chirp/TX/RX/sample 数据立方体。
4. 第一次 FFT：range FFT，从 ADC fast-time samples 得到距离维。
5. 第二次 FFT：Doppler FFT，从 slow-time chirps 得到速度维。
6. 第三次 FFT：angle FFT，从 virtual antenna array 得到角度维。
7. 在 range-Doppler-angle cube 中做阈值检测，输出点云。

说明：这里的代码按 IWR6843 + DCA1000 的常见复数 ADC 采集格式组织。真实实验中应把 `RadarConfig` 里的参数改成 mmWave Studio profile/chirp/frame 配置中的实际值。

## 1. 处理流程总览

TI mmWave Studio 采集到的 `.bin` 文件不是点云，而是 ADC 原始采样流。它通常来自 DCA1000 采集卡，把雷达芯片通过 LVDS 输出的 I/Q 数据写入磁盘。

从 raw bin 到点云，需要完成以下转换：

- **Raw ADC stream**：磁盘上的 `int16` 序列，保存 I/Q 采样值。
- **ADC cube**：按 chirp、RX、ADC sample 重排后的复数数据。
- **TDM-MIMO cube**：按 frame、loop、TX、RX、ADC sample 重排。
- **Range cube**：对 ADC sample 维做 range FFT。
- **Range-Doppler cube**：对每个 TX 对应的 chirp loop 维做 Doppler FFT。
- **Range-Doppler-Angle cube**：把 TX/RX 组合为 virtual antenna array，再做 angle FFT。
- **Point cloud**：对三维 FFT cube 做检测，把显著能量峰转成 range、velocity、angle、power。

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np

# 可选：只用于快速查看检测结果分布。
# 如果环境没有 matplotlib，可以注释掉后面的绘图 cell。
import matplotlib.pyplot as plt


## 2. 配置采集参数

下面这些参数必须和 mmWave Studio 中的 profile/chirp/frame 配置一致。最容易出错的是：

- `num_adc_samples`：每个 chirp 的 ADC 采样点数。
- `num_rx`：启用的接收天线数量，IWR6843 常见为 4 RX。
- `num_tx`：TDM-MIMO 中启用的发射天线数量，IWR6843 常见为 3 TX。
- `num_loops_per_frame`：每帧中每个 TX 重复的 chirp loop 数。
- `sample_rate_hz`：ADC sample rate。
- `slope_hz_per_s`：FMCW chirp slope。
- `chirp_period_s`：单个 chirp 的周期，通常约等于 idle time + ramp end time。

如果这些参数不匹配，数据 reshape 可能失败；即使 reshape 成功，range/velocity/angle 也会错。

In [ ]:
@dataclass
class RadarConfig:
    # 输入文件：替换成 mmWave Studio/DCA1000 导出的 bin 文件路径。
    adc_bin_path: Path = Path("adc_data_Raw_0.bin")

    # IWR6843 常见配置：3 TX x 4 RX = 12 virtual antennas。
    num_tx: int = 3
    num_rx: int = 4

    # 每个 chirp 的 ADC samples。必须与 mmWave Studio profileCfg 一致。
    num_adc_samples: int = 256

    # 每帧中每个 TX 的 chirp loops 数。TDM-MIMO 下每帧总 chirp 数为 num_tx * num_loops_per_frame。
    num_loops_per_frame: int = 128

    # FFT size 可以大于原始采样数，用于 zero padding。
    range_fft_size: int = 256
    doppler_fft_size: int = 128
    angle_fft_size: int = 64

    # 下面三个物理参数用于把 FFT bin 转成米、米每秒和角度。
    # 请用 mmWave Studio 中 profile/chirp 配置的真实值替换。
    sample_rate_hz: float = 5.0e6
    slope_hz_per_s: float = 60.0e12
    start_freq_hz: float = 60.0e9
    chirp_period_s: float = 60.0e-6

    # DCA1000/mmWave Studio 常见输出是 complex ADC data。
    # 如果采集为 real-only，需要单独处理；本文流程默认 complex。
    is_complex: bool = True


cfg = RadarConfig()
cfg


## 3. 读取 DCA1000 raw ADC bin

DCA1000 保存的是 `int16` LVDS stream。对 IWR6843 常见 complex 模式，可以按 TI 示例脚本 `readDCA1000.m` 的思路恢复复数样本：

- 每两个 int16 组成一个 complex sample 的 I/Q。
- LVDS stream 中 I/Q 和 lane 的交织顺序需要还原。
- 最终输出形状为 `[num_chirps_total, num_rx, num_adc_samples]`。

注意：不同 radar device、lane 配置和 capture mode 可能让交织顺序不同。如果结果出现明显错位、虚假峰或所有 RX 相位异常，需要回到 mmWave Studio/DCA1000 的 raw data format 检查 lane reorder。

In [ ]:
def read_dca1000_complex_bin(path: Path, num_rx: int, num_adc_samples: int) -> np.ndarray:
    """Read a common TI DCA1000 complex ADC bin file.

    Parameters
    ----------
    path:
        Path to the raw bin exported by mmWave Studio/DCA1000.
    num_rx:
        Number of enabled RX antennas.
    num_adc_samples:
        Number of ADC samples per chirp.

    Returns
    -------
    adc_cube:
        Complex array with shape [num_chirps_total, num_rx, num_adc_samples].

    Notes
    -----
    This follows the common DCA1000 complex-data pattern used by TI examples:
    int16 stream -> complex LVDS samples -> chirp/RX/sample cube.

    For complex data, every complex sample has I and Q components. The raw stream
    is often arranged in groups that TI's reference parser reconstructs as:
        sample_0 = raw[i]     + 1j * raw[i + 2]
        sample_1 = raw[i + 1] + 1j * raw[i + 3]
    This is why the conversion below steps through the int16 array in groups of 4.
    """
    raw = np.fromfile(path, dtype=np.int16)
    if raw.size == 0:
        raise ValueError(f"Empty ADC bin file: {path}")

    if raw.size % 4 != 0:
        raise ValueError(
            f"Raw int16 count {raw.size} is not divisible by 4. "
            "Check whether the file is truncated or uses a different capture format."
        )

    # Recover complex LVDS stream. This vectorized form is equivalent to the common
    # TI readDCA1000 loop that consumes raw[i:i+4] and produces two complex samples.
    raw4 = raw.reshape(-1, 4)
    complex_stream = np.empty(raw4.shape[0] * 2, dtype=np.complex64)
    complex_stream[0::2] = raw4[:, 0].astype(np.float32) + 1j * raw4[:, 2].astype(np.float32)
    complex_stream[1::2] = raw4[:, 1].astype(np.float32) + 1j * raw4[:, 3].astype(np.float32)

    samples_per_chirp_all_rx = num_rx * num_adc_samples
    if complex_stream.size % samples_per_chirp_all_rx != 0:
        raise ValueError(
            "Complex sample count does not fit num_rx * num_adc_samples. "
            f"complex_samples={complex_stream.size}, "
            f"num_rx={num_rx}, num_adc_samples={num_adc_samples}. "
            "Update RadarConfig to match the mmWave Studio capture settings."
        )

    num_chirps_total = complex_stream.size // samples_per_chirp_all_rx

    # First reshape into [chirp, rx * sample]. Inside each chirp, samples are grouped
    # by RX channel in the common TI parser output.
    lvds = complex_stream.reshape(num_chirps_total, samples_per_chirp_all_rx)

    # Final ADC cube: one chirp contains num_rx receive channels, and each RX channel
    # contains num_adc_samples fast-time ADC samples.
    adc_cube = lvds.reshape(num_chirps_total, num_rx, num_adc_samples)
    return adc_cube


## 4. 重排为 TDM-MIMO frame cube

IWR6843 常见配置会启用多个 TX 做 TDM-MIMO。chirp 顺序通常是：

```text
frame 0:
  loop 0: TX0 chirp, TX1 chirp, TX2 chirp
  loop 1: TX0 chirp, TX1 chirp, TX2 chirp
  ...
frame 1:
  loop 0: TX0 chirp, TX1 chirp, TX2 chirp
  ...
```

因此 raw ADC cube `[total_chirp, rx, sample]` 可以重排为：

```text
[frame, loop, tx, rx, sample]
```

如果 chirpCfg 的 TX 顺序不是 TX0/TX1/TX2 轮询，需要在这里调整 TX 维的顺序。

In [ ]:
def reshape_tdm_mimo_frames(adc_cube: np.ndarray, cfg: RadarConfig) -> np.ndarray:
    """Reshape [total_chirp, rx, sample] into [frame, loop, tx, rx, sample]."""
    chirps_per_frame = cfg.num_tx * cfg.num_loops_per_frame
    num_complete_frames = adc_cube.shape[0] // chirps_per_frame

    if num_complete_frames == 0:
        raise ValueError(
            "Not enough chirps for one complete frame. "
            f"total_chirps={adc_cube.shape[0]}, chirps_per_frame={chirps_per_frame}."
        )

    dropped_chirps = adc_cube.shape[0] - num_complete_frames * chirps_per_frame
    if dropped_chirps:
        print(f"Dropping {dropped_chirps} incomplete chirps at the end of the file.")

    adc_cube = adc_cube[: num_complete_frames * chirps_per_frame]

    # Shape explanation:
    #   frame: capture frame index
    #   loop: repeated slow-time chirps for Doppler FFT
    #   tx: TDM-MIMO transmitter index
    #   rx: physical receiver index
    #   sample: fast-time ADC sample index for range FFT
    frame_cube = adc_cube.reshape(
        num_complete_frames,
        cfg.num_loops_per_frame,
        cfg.num_tx,
        cfg.num_rx,
        cfg.num_adc_samples,
    )
    return frame_cube


## 5. 第一次 FFT：Range FFT

FMCW 雷达中，目标距离会反映为 beat frequency。对每个 chirp 的 ADC fast-time samples 做 FFT，可以得到 range bins。

输入：

```text
[loop, tx, rx, sample]
```

输出：

```text
[loop, tx, rx, range_bin]
```

这里使用 Hann window 降低旁瓣。实际系统中还可能加入 DC removal、静态杂波消除、RX 通道校准等步骤。

In [ ]:
def range_fft(frame: np.ndarray, cfg: RadarConfig) -> np.ndarray:
    """Run range FFT along the ADC sample axis.

    Parameters
    ----------
    frame:
        One frame with shape [loop, tx, rx, sample].

    Returns
    -------
    range_cube:
        Complex range cube with shape [loop, tx, rx, range_bin].
    """
    window = np.hanning(cfg.num_adc_samples).astype(np.float32)

    # Apply window on the fast-time ADC sample axis, then FFT into range bins.
    windowed = frame * window[None, None, None, :]
    cube = np.fft.fft(windowed, n=cfg.range_fft_size, axis=-1)

    # Only positive beat frequencies are physically useful for normal FMCW range.
    # Keeping all bins can be useful for debugging; here we keep the full FFT and
    # handle valid range bins during point extraction.
    return cube


## 6. 第二次 FFT：Doppler FFT

目标速度会反映为同一 TX 连续 chirp 之间的相位变化。对 slow-time loop 维做 FFT，可以得到 Doppler bins。

输入：

```text
[loop, tx, rx, range_bin]
```

输出：

```text
[doppler_bin, tx, rx, range_bin]
```

TDM-MIMO 下，同一 TX 的 chirp 间隔不是单个 chirp period，而是 `num_tx * chirp_period_s`。后面把 Doppler bin 转速度时会用到这个间隔。

In [ ]:
def doppler_fft(range_cube: np.ndarray, cfg: RadarConfig) -> np.ndarray:
    """Run Doppler FFT along the slow-time loop axis."""
    window = np.hanning(cfg.num_loops_per_frame).astype(np.float32)

    # Slow-time window reduces Doppler sidelobes.
    windowed = range_cube * window[:, None, None, None]

    # FFT over loop axis. fftshift moves zero Doppler to the center.
    cube = np.fft.fft(windowed, n=cfg.doppler_fft_size, axis=0)
    cube = np.fft.fftshift(cube, axes=0)
    return cube


## 7. 第三次 FFT：Angle FFT

TDM-MIMO 会把 `num_tx * num_rx` 个 TX/RX 通道组合成 virtual antenna array。目标角度会表现为 virtual antenna 维上的相位梯度。

输入：

```text
[doppler_bin, tx, rx, range_bin]
```

先重排为：

```text
[doppler_bin, virtual_antenna, range_bin]
```

再对 virtual antenna 维做 angle FFT，输出：

```text
[doppler_bin, angle_bin, range_bin]
```

重要：IWR6843 的真实 virtual array 可能包含 azimuth/elevation 通道，且 TX/RX 的物理位置不是简单线性排列。严格成像时应加入 antenna order、phase calibration 和 azimuth/elevation 分离。这里给出的是三次 FFT pipeline 的清晰工程骨架。

In [ ]:
def angle_fft(doppler_cube: np.ndarray, cfg: RadarConfig) -> np.ndarray:
    """Run angle FFT over the virtual antenna axis.

    Returns a complex cube with shape [doppler_bin, angle_bin, range_bin].
    """
    num_virtual_ant = cfg.num_tx * cfg.num_rx

    # Combine TX and RX into a virtual antenna dimension.
    # Input:  [doppler, tx, rx, range]
    # Output: [doppler, virtual_ant, range]
    virtual_cube = doppler_cube.reshape(
        cfg.doppler_fft_size,
        num_virtual_ant,
        cfg.range_fft_size,
    )

    # Window virtual array before angle FFT to reduce angular sidelobes.
    window = np.hanning(num_virtual_ant).astype(np.float32)
    windowed = virtual_cube * window[None, :, None]

    # FFT over virtual antenna axis. fftshift moves broadside near the center bin.
    cube = np.fft.fft(windowed, n=cfg.angle_fft_size, axis=1)
    cube = np.fft.fftshift(cube, axes=1)
    return cube


## 8. FFT bin 到物理量的换算

检测点最终需要从 FFT bin 转成工程上可解释的物理量：

- range bin -> meters
- Doppler bin -> meters/second
- angle bin -> degree

换算依赖 radar profile 参数。下面的公式使用常见 FMCW 近似：

- `range = c * beat_frequency / (2 * slope)`
- `velocity = doppler_frequency * wavelength / 2`
- 对半波长间距 ULA，`sin(theta) ~= 2 * spatial_bin / angle_fft_size`


In [ ]:
LIGHT_SPEED = 299_792_458.0


def range_axis_m(cfg: RadarConfig) -> np.ndarray:
    """Return range value in meters for each range FFT bin."""
    beat_freq = np.arange(cfg.range_fft_size) * cfg.sample_rate_hz / cfg.range_fft_size
    return LIGHT_SPEED * beat_freq / (2.0 * cfg.slope_hz_per_s)


def velocity_axis_mps(cfg: RadarConfig) -> np.ndarray:
    """Return velocity value in m/s for each shifted Doppler FFT bin."""
    wavelength = LIGHT_SPEED / cfg.start_freq_hz

    # Same-TX chirps are separated by num_tx chirp periods in TDM-MIMO.
    slow_time_step = cfg.num_tx * cfg.chirp_period_s
    doppler_freq = np.fft.fftshift(np.fft.fftfreq(cfg.doppler_fft_size, d=slow_time_step))
    return doppler_freq * wavelength / 2.0


def angle_axis_deg(cfg: RadarConfig) -> np.ndarray:
    """Return approximate azimuth angle for each shifted angle FFT bin.

    This assumes a uniform linear virtual array with lambda/2 spacing. For full
    IWR6843 azimuth/elevation processing, replace this with calibrated antenna geometry.
    """
    shifted_bins = np.arange(cfg.angle_fft_size) - cfg.angle_fft_size // 2
    sin_theta = 2.0 * shifted_bins / cfg.angle_fft_size
    sin_theta = np.clip(sin_theta, -1.0, 1.0)
    return np.degrees(np.arcsin(sin_theta))


ranges_m = range_axis_m(cfg)
velocities_mps = velocity_axis_mps(cfg)
angles_deg = angle_axis_deg(cfg)

print("range bins:", ranges_m[:5], "...")
print("velocity bins:", velocities_mps[:5], "...")
print("angle bins:", angles_deg[:5], "...")


## 9. 从三维 FFT cube 中检测点

工业级 radar pipeline 通常使用 CFAR、静态杂波消除、peak grouping、SNR filtering、multi-frame tracking 等步骤。这里为了把主流程写清楚，先用一个简单的 median noise floor + threshold 方法：

1. 计算 `angle_cube` 的功率。
2. 用中位数估计噪声底。
3. 保留超过噪声底一定 dB 的 cell。
4. 按功率排序，输出 top-K 点。

后续如果要用于论文级复现或工程展示，建议把这一段替换为 CA-CFAR/OS-CFAR，并补充 peak grouping。

In [ ]:
def detect_points_from_angle_cube(
    angle_cube: np.ndarray,
    cfg: RadarConfig,
    threshold_db_above_median: float = 18.0,
    max_points: int = 2048,
    min_range_m: float = 0.15,
    max_range_m: float | None = None,
) -> np.ndarray:
    """Detect point candidates from [doppler, angle, range] FFT cube.

    Returns a structured NumPy array with columns:
        range_m, velocity_mps, angle_deg, power_db, doppler_bin, angle_bin, range_bin
    """
    power = np.abs(angle_cube) ** 2
    power_db = 10.0 * np.log10(power + 1e-12)

    ranges = range_axis_m(cfg)
    velocities = velocity_axis_mps(cfg)
    angles = angle_axis_deg(cfg)

    valid_range_mask = ranges >= min_range_m
    if max_range_m is not None:
        valid_range_mask &= ranges <= max_range_m

    # Apply valid range mask by setting invalid bins to negative infinity.
    masked_power_db = power_db.copy()
    masked_power_db[:, :, ~valid_range_mask] = -np.inf

    finite_values = masked_power_db[np.isfinite(masked_power_db)]
    if finite_values.size == 0:
        raise ValueError("No valid range bins remain after range filtering.")

    noise_floor_db = np.median(finite_values)
    threshold_db = noise_floor_db + threshold_db_above_median

    detection_mask = masked_power_db > threshold_db
    coords = np.argwhere(detection_mask)

    dtype = [
        ("range_m", "f4"),
        ("velocity_mps", "f4"),
        ("angle_deg", "f4"),
        ("power_db", "f4"),
        ("doppler_bin", "i4"),
        ("angle_bin", "i4"),
        ("range_bin", "i4"),
    ]

    if coords.size == 0:
        return np.zeros(0, dtype=dtype)

    scores = masked_power_db[detection_mask]
    top_idx = np.argsort(scores)[::-1][:max_points]
    coords = coords[top_idx]
    scores = scores[top_idx]

    points = np.zeros(coords.shape[0], dtype=dtype)
    for i, (doppler_bin, angle_bin, range_bin) in enumerate(coords):
        points[i] = (
            ranges[range_bin],
            velocities[doppler_bin],
            angles[angle_bin],
            scores[i],
            doppler_bin,
            angle_bin,
            range_bin,
        )

    return points


## 10. 一键处理一个 frame

把上面的步骤串起来，就是完整的 raw bin -> FFT cube -> point cloud pipeline。

如果当前目录没有 ADC bin 文件，下面的 cell 会提示你修改 `cfg.adc_bin_path`，不会产生结果。

In [ ]:
def process_one_frame(cfg: RadarConfig, frame_index: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """Read raw bin, run 3 FFTs, and return angle cube plus detected points."""
    if not cfg.adc_bin_path.exists():
        raise FileNotFoundError(
            f"ADC bin not found: {cfg.adc_bin_path}. "
            "Set cfg.adc_bin_path to the file exported by mmWave Studio/DCA1000."
        )

    # Step 1: raw int16 stream -> complex ADC cube [total_chirp, rx, sample]
    adc_cube = read_dca1000_complex_bin(
        cfg.adc_bin_path,
        num_rx=cfg.num_rx,
        num_adc_samples=cfg.num_adc_samples,
    )
    print("ADC cube:", adc_cube.shape, "[total_chirp, rx, sample]")

    # Step 2: ADC cube -> TDM-MIMO frame cube [frame, loop, tx, rx, sample]
    frame_cube = reshape_tdm_mimo_frames(adc_cube, cfg)
    print("Frame cube:", frame_cube.shape, "[frame, loop, tx, rx, sample]")

    if frame_index >= frame_cube.shape[0]:
        raise IndexError(f"frame_index={frame_index} but only {frame_cube.shape[0]} frames are available.")

    frame = frame_cube[frame_index]

    # Step 3: first FFT, ADC samples -> range bins
    r_cube = range_fft(frame, cfg)
    print("Range cube:", r_cube.shape, "[loop, tx, rx, range]")

    # Step 4: second FFT, slow-time loops -> Doppler bins
    rd_cube = doppler_fft(r_cube, cfg)
    print("Range-Doppler cube:", rd_cube.shape, "[doppler, tx, rx, range]")

    # Step 5: third FFT, virtual antenna -> angle bins
    rda_cube = angle_fft(rd_cube, cfg)
    print("Range-Doppler-Angle cube:", rda_cube.shape, "[doppler, angle, range]")

    # Step 6: threshold detection -> point cloud candidates
    points = detect_points_from_angle_cube(rda_cube, cfg)
    print("Detected points:", points.shape[0])
    return rda_cube, points


In [ ]:
# 使用方式：
# 1. 把 cfg.adc_bin_path 改成 mmWave Studio/DCA1000 导出的 bin 文件。
# 2. 把 RadarConfig 中的 profile/chirp/frame 参数改成真实采集配置。
# 3. 运行下面两行。
#
# cfg.adc_bin_path = Path("/path/to/adc_data_Raw_0.bin")
# angle_cube, points = process_one_frame(cfg, frame_index=0)


## 11. 简单可视化

下面的图不是最终论文图，只是用来快速检查 pipeline 是否工作：

- range-Doppler heatmap：检查距离/速度维是否有明显目标峰。
- point scatter：把检测点投到 angle-range 平面。

In [ ]:
def plot_range_doppler(angle_cube: np.ndarray, cfg: RadarConfig) -> None:
    """Plot a range-Doppler heatmap by max-pooling over angle bins."""
    rd_power = np.max(np.abs(angle_cube) ** 2, axis=1)
    rd_power_db = 10.0 * np.log10(rd_power + 1e-12)

    ranges = range_axis_m(cfg)
    velocities = velocity_axis_mps(cfg)

    plt.figure(figsize=(10, 5))
    plt.imshow(
        rd_power_db,
        aspect="auto",
        origin="lower",
        extent=[ranges[0], ranges[-1], velocities[0], velocities[-1]],
        cmap="viridis",
    )
    plt.xlabel("Range (m)")
    plt.ylabel("Velocity (m/s)")
    plt.title("Range-Doppler Heatmap")
    plt.colorbar(label="Power (dB)")
    plt.tight_layout()


def plot_points_angle_range(points: np.ndarray) -> None:
    """Plot detected points in angle-range space."""
    if points.size == 0:
        print("No points to plot.")
        return

    plt.figure(figsize=(8, 5))
    scatter = plt.scatter(
        points["angle_deg"],
        points["range_m"],
        c=points["power_db"],
        s=8,
        cmap="magma",
    )
    plt.xlabel("Angle (deg)")
    plt.ylabel("Range (m)")
    plt.title("Detected Radar Points")
    plt.colorbar(scatter, label="Power (dB)")
    plt.tight_layout()


In [ ]:
# 如果已经运行 process_one_frame 得到了 angle_cube 和 points，可以取消下面注释：
# plot_range_doppler(angle_cube, cfg)
# plot_points_angle_range(points)


## 12. 后续可增强的工程点

- **参数解析**：从 mmWave Studio 的配置文件自动读取 profile/chirp/frame 参数，减少手填错误。
- **静态杂波消除**：对 slow-time 维去均值或高通滤波，降低静态背景反射。
- **CFAR 检测**：替换简单阈值，使用 CA-CFAR/OS-CFAR 提高检测稳定性。
- **天线校准**：加入 virtual antenna order、相位补偿和幅度补偿。
- **3D 角度估计**：针对 IWR6843 azimuth/elevation virtual array 分别估计方位角和俯仰角。
- **点云后处理**：增加 peak grouping、SNR 过滤、multi-frame tracking 和人体 mesh/gesture 特征提取。
- **深度学习接口**：把每帧点云或 radar cube 输出为 PointNet/LSTM/Transformer 可读取的数据格式。